<a href="https://colab.research.google.com/github/irfanali11/sensor-fusion-robustness-study/blob/main/sensor_fusion_robustness_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision opencv-python scipy numpy matplotlib scikit-learn -q

In [ ]:
!wget "http://www.utdallas.edu/~kehtar/UTD-MAD/RGB.zip" -O rgb.zip
!wget "http://www.utdallas.edu/~kehtar/UTD-MAD/Inertial.zip" -O inertial.zip

!mkdir -p data/rgb data/inertial
!unzip -q rgb.zip -d data/rgb
!unzip -q inertial.zip -d data/inertial

!ls data/rgb | head -5
!ls data/inertial | head -5

--2026-09-04 22:05:14--  http://www.utdallas.edu/~kehtar/UTD-MAD/RGB.zip
Resolving www.utdallas.edu (www.utdallas.edu)... 3.21.250.42, 3.133.32.155
Connecting to www.utdallas.edu (www.utdallas.edu)|3.21.250.42|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.utdallas.edu/~kehtar/UTD-MAD/RGB.zip [following]
--2026-09-04 22:05:15--  https://www.utdallas.edu/~kehtar/UTD-MAD/RGB.zip
Connecting to www.utdallas.edu (www.utdallas.edu)|3.21.250.42|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://personal.utdallas.edu/~kehtar/UTD-MAD/RGB.zip [following]
--2026-09-04 22:05:16--  https://personal.utdallas.edu/~kehtar/UTD-MAD/RGB.zip
Resolving personal.utdallas.edu (personal.utdallas.edu)... 129.110.46.112
Connecting to personal.utdallas.edu (personal.utdallas.edu)|129.110.46.112|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1106885499 (1.0G) [application/zip]
Saving to: ‘rgb.zip’

In [ ]:
!ls data/rgb/RGB | head -5
!ls data/inertial/Inertial | head -5

In [ ]:
!ls data/rgb/RGB | wc -l
!ls data/inertial/Inertial | wc -l

In [ ]:
import os
import cv2
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
def load_video_frames(video_path, num_frames=16, resize=(112, 112)):
    """Load a fixed number of evenly-spaced frames from a video, resized."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return np.zeros((num_frames, *resize, 3), dtype=np.float32)

    idxs = np.linspace(0, max(total - 1, 0), num_frames).astype(int)
    idx_set = set(idxs.tolist())
    current = 0
    grabbed = {}
    while cap.isOpened() and len(grabbed) < len(idx_set):
        ret, frame = cap.read()
        if not ret:
            break
        if current in idx_set:
            frame = cv2.resize(frame, resize)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            grabbed[current] = frame
        current += 1
    cap.release()

    frames = [grabbed.get(i, np.zeros((*resize, 3), dtype=np.float32)) for i in idxs]
    return np.stack(frames, axis=0)  # (num_frames, H, W, 3)


def load_inertial(mat_path, seq_len=100):
    """Load IMU data from .mat file, pad/truncate to fixed sequence length."""
    mat = sio.loadmat(mat_path)
    data = mat["d_iner"].astype(np.float32)  # shape (T, 6): accel(3)+gyro(3)
    if data.shape[0] >= seq_len:
        data = data[:seq_len]
    else:
        pad = np.zeros((seq_len - data.shape[0], data.shape[1]), dtype=np.float32)
        data = np.concatenate([data, pad], axis=0)
    return data  # (seq_len, 6)


def parse_action_label(filename):
    """'a1_s2_t3_color.avi' -> action label 0 (zero-indexed)."""
    base = os.path.basename(filename)
    action_str = base.split("_")[0]  # 'a1'
    return int(action_str.replace("a", "")) - 1